In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load Gold layer tables
fact = spark.table("novacart_dev.gold.fact_order_items")
dim_products = spark.table("novacart_dev.gold.dim_products")
dim_customers = spark.table("novacart_dev.gold.dim_customers")
dim_dates = spark.table("novacart_dev.gold.dim_dates")
dim_orders = spark.table("novacart_dev.gold.dim_orders")

print("✓ Gold layer tables loaded successfully")
print(f"  - Fact table: {fact.count():,} line items")
print(f"  - Products: {dim_products.count():,}")
print(f"  - Customers: {dim_customers.count():,}")
print(f"  - Orders: {dim_orders.count():,}")

In [0]:
# Store KPI 1: Total Revenue in a table
# Only count completed orders
total_revenue_df = fact.filter(F.col("order_status_clean") == "completed") \
    .agg(
        F.sum("revenue_usd").alias("total_revenue_usd")
    )
total_revenue_df.write.mode("overwrite").saveAsTable("novacart_dev.gold.kpi_total_revenue")
print("kpi_total_revenue made successfully!")

In [0]:
# KPI 2: Revenue by Country
# Only count completed orders
revenue_by_country = fact.filter(F.col("order_status_clean") == "completed") \
    .groupBy("order_country") \
    .agg(
        F.sum("revenue_usd").alias("revenue_usd"),
        F.count("order_item_key").alias("order_items")
    ) \
    .orderBy(F.desc("revenue_usd"))

revenue_by_country.write.mode("overwrite").saveAsTable("novacart_dev.gold.kpi_revenue_by_country")
print("kpi_revenue_by_country made successfully!")

In [0]:
# KPI 3: Revenue by Channel
# Only count completed orders
revenue_by_channel = fact.filter(F.col("order_status_clean") == "completed") \
    .groupBy("channel") \
    .agg(
        F.sum("revenue_usd").alias("revenue_usd"),
        F.count("order_item_key").alias("order_items"),
        F.countDistinct("order_key").alias("orders")
    ) \
    .orderBy(F.desc("revenue_usd"))

revenue_by_channel.write.mode("overwrite").saveAsTable("novacart_dev.gold.kpi_revenue_by_channel")
print("kpi_revenue_by_channel made successfully!")

In [0]:
# KPI 3b: Revenue by Product Currency
# Shows original currency totals vs USD-converted totals
# Only count completed orders
revenue_by_currency = fact.filter(F.col("order_status_clean") == "completed") \
    .groupBy("product_currency") \
    .agg(
        F.sum("line_total").alias("total_local_currency"),
        F.sum("revenue_usd").alias("total_usd"),
        F.count("order_item_key").alias("order_items"),
        F.countDistinct("order_key").alias("orders")
    ) \
    .withColumn(
        "avg_exchange_rate",
        F.round(F.col("total_usd") / F.col("total_local_currency"), 4)
    ) \
    .orderBy(F.desc("total_usd"))

revenue_by_currency.write.mode("overwrite").saveAsTable("novacart_dev.gold.kpi_revenue_by_currency")
print("kpi_revenue_by_currency created successfully!")
print("\nThis KPI demonstrates currency conversion from product local currency to USD")

In [0]:
# KPI 4: Completed Order Count
completed_orders = fact.filter(F.col("order_status_clean") == "completed") \
    .select("order_key") \
    .distinct() \
    .count()

# Store KPI 4: Completed Order Count in a table
spark.createDataFrame(
    [(completed_orders,)],
    ["completed_orders"]
).write.mode("overwrite").saveAsTable("novacart_dev.gold.kpi_completed_order_count")

print("kpi_completed_order_count made successfully!")

In [0]:
# KPI 5: Completed Order Rate
total_orders = fact.select("order_key").distinct().count()
completed_order_rate = (completed_orders / total_orders * 100) if total_orders > 0 else 0

# Store KPI 5: Completed Order Rate in a table
spark.createDataFrame(
    [(total_orders, completed_orders, completed_order_rate)],
    ["total_orders", "completed_orders", "completed_order_rate"]
).write.mode("overwrite").saveAsTable("novacart_dev.gold.kpi_completed_order_rate")

print("kpi_completed_order_rate created successfully!")
# display(spark.table("novacart_dev.gold.kpi_completed_order_rate"))

In [0]:
# KPI 6: Average Order Value (AOV)
# Only count completed orders
aov_df = fact.filter(F.col("order_status_clean") == "completed") \
    .groupBy("order_key") \
    .agg(F.sum("revenue_usd").alias("order_total")) \
    .agg(
        F.avg("order_total").alias("avg_order_value"),
        F.min("order_total").alias("min_order_value"),
        F.max("order_total").alias("max_order_value")
    )

aov_df.write.mode("overwrite").saveAsTable("novacart_dev.gold.kpi_average_order_value")
print("kpi_average_order_value made successfully!")
# display(spark.table("novacart_dev.gold.kpi_average_order_value"))

In [0]:
# KPI 7: Top 5 Products by Revenue
# Filter out null product keys (orphan products) and only count completed orders
top_products = fact.filter(
        (F.col("product_key").isNotNull()) & 
        (F.col("order_status_clean") == "completed")
    ) \
    .join(dim_products, fact.product_key == dim_products.product_key, "left") \
    .groupBy(fact.product_key, dim_products.product_name, dim_products.category) \
    .agg(
        F.sum("revenue_usd").alias("total_revenue"),
        F.sum("quantity").alias("total_quantity_sold"),
        F.countDistinct(fact.order_key).alias("order_count")
    ) \
    .orderBy(F.desc("total_revenue")) \
    .limit(5)

top_products.write.mode("overwrite").saveAsTable("novacart_dev.gold.kpi_top_products")
print("kpi_top_products made successfully!")
# display(spark.table("novacart_dev.gold.kpi_top_products"))

In [0]:
# KPI 8: Active Customers Count
# Customers who have placed at least one completed order
active_customers = fact.filter(F.col("order_status_clean") == "completed") \
    .select("customer_key") \
    .distinct() \
    .count()

total_customers = dim_customers.count()
active_rate = (active_customers / total_customers * 100) if total_customers > 0 else 0

# Store KPI 8: Active Customers Count in a table
spark.createDataFrame(
    [(active_customers, total_customers, active_rate)],
    ["active_customers", "total_customers", "active_rate"]
).write.mode("overwrite").saveAsTable("novacart_dev.gold.kpi_active_customers_count")

print("kpi_active_customers_count made successfully!")
# display(spark.table("novacart_dev.gold.kpi_active_customers_count"))

In [0]:
# KPI 9: Customer Acquisition by Month
# Filter out customers with null registration dates
customer_acquisition = dim_customers \
    .filter(
        F.col("registration_year").isNotNull() & 
        F.col("registration_month").isNotNull()
    ) \
    .groupBy("registration_year", "registration_month") \
    .agg(
        F.count("customer_key").alias("new_customers")
    ) \
    .withColumn(
        "year_month",
        F.concat(
            F.col("registration_year").cast("string"),
            F.lit("-"),
            F.lpad(F.col("registration_month").cast("string"), 2, "0")
        )
    ) \
    .orderBy("registration_year", "registration_month")

customer_acquisition.write.mode("overwrite").saveAsTable("novacart_dev.gold.kpi_customer_acquisition")
print("kpi_customer_acquisition made successfully!")
# display(spark.table("novacart_dev.gold.kpi_customer_acquisition"))

In [0]:
# KPI 10: Data Quality Score
# Calculate percentage of records without data quality issues
total_records = fact.count()

# Get all DQ columns dynamically
dq_columns = [col for col in fact.columns if col.startswith('dq_')]
print(f"Checking {len(dq_columns)} data quality flags...")

# Build dynamic filter condition for records with ANY DQ issue
filter_conditions = [F.col(col) == 1 for col in dq_columns]
records_with_issues = fact.filter(F.expr(' OR '.join([f'({col} = 1)' for col in dq_columns]))).count()

clean_records = total_records - records_with_issues
data_quality_score = (clean_records / total_records * 100) if total_records > 0 else 0

# Breakdown by issue type - aggregate all DQ columns
issue_counts = {}
for col in dq_columns:
    count = fact.filter(F.col(col) == 1).count()
    if count > 0:
        issue_counts[col] = count

# Create detailed breakdown DataFrame
breakdown_data = [(dq_flag, count, round(count/total_records*100, 2)) 
                  for dq_flag, count in sorted(issue_counts.items(), key=lambda x: x[1], reverse=True)]

breakdown_df = spark.createDataFrame(
    breakdown_data,
    ["dq_flag", "affected_records", "percentage"]
)

# Store the detailed breakdown
breakdown_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("novacart_dev.gold.kpi_dq_breakdown")

# Store KPI 10: Data Quality Score summary
spark.createDataFrame(
    [(total_records, clean_records, records_with_issues, data_quality_score, len(dq_columns))],
    ["total_records", "clean_records", "records_with_issues", "data_quality_score", "total_dq_checks"]
).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("novacart_dev.gold.kpi_data_quality_score")

print("\n" + "="*80)
print("KPI 10: DATA QUALITY SCORE")
print("="*80)
print(f"Total Records: {total_records:,}")
print(f"Clean Records: {clean_records:,} ({data_quality_score:.2f}%)")
print(f"Records with Issues: {records_with_issues:,}")
print(f"Total DQ Checks: {len(dq_columns)}")
print("\nTop Issues Found:")
for dq_flag, count in sorted(issue_counts.items(), key=lambda x: x[1], reverse=True)[:5]:
    pct = count/total_records*100
    print(f"  - {dq_flag}: {count:,} ({pct:.2f}%)")
print("="*80)
print("\n✓ kpi_data_quality_score and kpi_dq_breakdown tables created successfully!")

In [0]:
# Display detailed breakdown of all DQ issues
dq_breakdown = spark.table("novacart_dev.gold.kpi_dq_breakdown")
print("Data Quality Issues - Detailed Breakdown:")
print("="*80)
display(dq_breakdown)